# Machine Learning techniques for Fraud Detection in the Big Data





This Project has used the IBM Credit Card dataset of 2.3 GB.

Obejectives of these Project are by following:
- Performing the Data Chunking Process into the HDFS.
- Do Exploratory Analysis of the Fraud Detection Data
- Apply appropreate Machine learning Model to Predict the Fraud Detection.

This Project has used the two main techniques:
- HDFS (Hadoop Distributed File System) for choping the dataset.
- PySpark for the analysis of the data.
- Pandas for the Manipulation of the data.
- Matplotlib and Seaborn for the visualisation.


**Note: This project used Python 3.13 for runtime, Apache Spark version 3.3.6, and Java 11.**

## Part 1: HDFS

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
# Step 1: Install Java 11.
# The -q option keeps the output shorter.
# The -y option automatically accepts the installation request.

!apt-get update -qq
!apt-get install -y -qq openjdk-11-jdk-headless > /dev/null

print("Java installation completed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Java installation completed.


In [10]:
# Step 2: Define Hadoop and Java folder locations.

import os
from pathlib import Path

HADOOP_VERSION = "3.3.6"
HADOOP_HOME = f"/content/hadoop-{HADOOP_VERSION}"
JAVA_HOME = "/usr/lib/jvm/java-11-openjdk-amd64"

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["HADOOP_CONF_DIR"] = f"{HADOOP_HOME}/etc/hadoop"
os.environ["HADOOP_COMMON_HOME"] = HADOOP_HOME
os.environ["HADOOP_HDFS_HOME"] = HADOOP_HOME
os.environ["PATH"] = (
    f"{HADOOP_HOME}/bin:"
    f"{HADOOP_HOME}/sbin:"
    + os.environ["PATH"]
)

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])

JAVA_HOME = /usr/lib/jvm/java-11-openjdk-amd64
HADOOP_HOME = /content/hadoop-3.3.6


In [11]:
# Step 3: Download and extract Hadoop only when required.

archive = f"/content/hadoop-{HADOOP_VERSION}.tar.gz"

if not Path(HADOOP_HOME).exists():
    print("Downloading Hadoop...")
    !wget -q "https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VERSION}/hadoop-{HADOOP_VERSION}.tar.gz" -O "$archive"

    print("Extracting Hadoop...")
    !tar -xzf "$archive" -C /content

    print("Hadoop downloaded and extracted.")
else:
    print("Hadoop is already available. Download skipped.")



Hadoop is already available. Download skipped.


In [12]:

namenode_dir = Path("/content/hadoop_data/namenode")
datanode_dir = Path("/content/hadoop_data/datanode")

namenode_dir.mkdir(parents=True, exist_ok=True)
datanode_dir.mkdir(parents=True, exist_ok=True)

print("NameNode storage folder:", namenode_dir)
print("DataNode storage folder:", datanode_dir)

NameNode storage folder: /content/hadoop_data/namenode
DataNode storage folder: /content/hadoop_data/datanode


In [13]:
# Step 6: Write the core-site.xml configuration file.

core_site = """<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>
"""

core_site_path = Path(HADOOP_HOME) / "etc/hadoop/core-site.xml"
core_site_path.write_text(core_site)

print(core_site_path.read_text())

<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>



In [14]:
# Step 7: Write the hdfs-site.xml configuration file.

hdfs_site = f"""<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>

    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file:{namenode_dir}</value>
    </property>

    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file:{datanode_dir}</value>
    </property>

    <property>
        <name>dfs.permissions.enabled</name>
        <value>false</value>
    </property>
</configuration>
"""

hdfs_site_path = Path(HADOOP_HOME) / "etc/hadoop/hdfs-site.xml"
hdfs_site_path.write_text(hdfs_site)

print(hdfs_site_path.read_text())

<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>

    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file:/content/hadoop_data/namenode</value>
    </property>

    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file:/content/hadoop_data/datanode</value>
    </property>

    <property>
        <name>dfs.permissions.enabled</name>
        <value>false</value>
    </property>
</configuration>



In [15]:
# Step 8: Add the Java location to Hadoop's environment file.

hadoop_env_path = Path(HADOOP_HOME) / "etc/hadoop/hadoop-env.sh"
hadoop_env_text = hadoop_env_path.read_text()
java_line = f"export JAVA_HOME={JAVA_HOME}"

if java_line not in hadoop_env_text:
    with hadoop_env_path.open("a") as file:
        file.write("\n" + java_line + "\n")

print("Added to hadoop-env.sh:")
print(java_line)

Added to hadoop-env.sh:
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64


In [16]:
# Step 9: Format the NameNode only on the first execution.

namenode_current = namenode_dir / "current"

if not namenode_current.exists():
    print("Formatting the NameNode...")
    !hdfs namenode -format -force -nonInteractive > /content/namenode_format.log 2>&1
    print("NameNode formatting completed.")
else:
    print("NameNode is already formatted. Formatting skipped.")

if Path("/content/namenode_format.log").exists():
    !tail -n 8 /content/namenode_format.log

NameNode is already formatted. Formatting skipped.
2026-09-20 09:53:25,600 INFO namenode.NNStorageRetentionManager: Going to retain 1 images with txid >= 0
2026-09-20 09:53:25,646 INFO namenode.FSNamesystem: Stopping services started for active state
2026-09-20 09:53:25,646 INFO namenode.FSNamesystem: Stopping services started for standby state
2026-09-20 09:53:25,652 INFO namenode.FSImage: FSImageSaver clean checkpoint: txid=0 when meet shutdown.
2026-09-20 09:53:25,653 INFO namenode.NameNode: SHUTDOWN_MSG: 
/************************************************************
SHUTDOWN_MSG: Shutting down NameNode at 7f591164d9ac/172.28.0.12
************************************************************/


In [17]:
# Step 10: Stop older HDFS processes and start fresh services.

!hdfs --daemon stop secondarynamenode 2>/dev/null
!hdfs --daemon stop datanode 2>/dev/null
!hdfs --daemon stop namenode 2>/dev/null

!hdfs --daemon start namenode
!hdfs --daemon start datanode
!hdfs --daemon start secondarynamenode

print("HDFS services have been started.")

HDFS services have been started.


In [18]:
# Step 11: Check the running Java processes.

!jps

11600 NameNode
11765 Jps
11660 DataNode
11725 SecondaryNameNode
10783 SparkSubmit


In [19]:
# Step 12: Confirm that HDFS is responding.

!hdfs dfs -ls /

Found 1 items
drwxr-xr-x   - root supergroup          0 2026-09-20 10:10 /credit


In [20]:
# mount drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/credit_card_transactions-ibm_v2_csv.zip"
extract_path = "/content/extracted_folder"

os.makedirs(extract_path, exist_ok=True)
try:
  with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

  print(os.listdir(extract_path))
except Exception as e:
  print(f"An error occurred: {e}")

['credit_card_transactions-ibm_v2.csv']


In [22]:
# import os

# zip_path = "/content/credit_card_transactions-ibm_v2_csv.zip"

# print("Exists:", os.path.exists(zip_path))

# if os.path.exists(zip_path):
#     print("File size:", os.path.getsize(zip_path), "bytes")

#     with open(zip_path, "rb") as f:
#         first_bytes = f.read(20)

#     print("First 20 bytes:", first_bytes)
# !file "/content/credit_card_transactions-ibm_v2_csv.zip"
# !unzip -t "/content/credit_card_transactions-ibm_v2_csv.zip"

In [23]:
local_file = "/content/extracted_folder/credit_card_transactions-ibm_v2.csv"

In [24]:
# Step 14: Display the local CSV file as plain text.
# displaying first five rows
!head -n 5 /content/extracted_folder/credit_card_transactions-ibm_v2.csv
#!cat /content/extracted_folder/credit_card_transactions-ibm_v2.csv

User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,,No
0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,,No
0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,,No
0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,,No


In [25]:
# Step 15: Create the credit directory in HDFS.
!hdfs dfs -mkdir -p /credit
!hdfs dfs -ls /

Found 1 items
drwxr-xr-x   - root supergroup          0 2026-09-20 10:10 /credit


In [26]:
# Step 16: Remove an old HDFS copy if one exists.
!hdfs dfs -rm -f /credit/credit_card_transactions-ibm_v2.csv

# Upload the local CSV file into HDFS.
!hdfs dfs -put /content/extracted_folder/credit_card_transactions-ibm_v2.csv /credit/

# Confirm the file is present.
!hdfs dfs -ls /credit

Deleted /credit/credit_card_transactions-ibm_v2.csv
Found 2 items
-rw-r--r--   1 root supergroup 2350744057 2026-09-20 10:23 /credit/credit_card_transactions-ibm_v2.csv
-rw-r--r--   1 root supergroup  208414209 2026-09-20 10:10 /credit/credit_transactions.parquet


In [27]:
# Step 17: Show the HDFS file name and size.

!hdfs dfs -stat "File name: %n | Size: %b bytes" /credit/credit_card_transactions-ibm_v2.csv

File name: credit_card_transactions-ibm_v2.csv | Size: 2350744057 bytes


In [28]:
# Step 18: Inspect the file blocks and their DataNode locations.

!hdfs fsck /credit/credit_card_transactions-ibm_v2.csv -files -blocks -locations

Connecting to namenode via http://localhost:9870/fsck?ugi=root&files=1&blocks=1&locations=1&path=%2Fcredit%2Fcredit_card_transactions-ibm_v2.csv
FSCK started by root (auth:SIMPLE) from /127.0.0.1 for path /credit/credit_card_transactions-ibm_v2.csv at Sun Sep 20 10:23:28 UTC 2026

/credit/credit_card_transactions-ibm_v2.csv 2350744057 bytes, replicated: replication=1, 18 block(s):  OK
0. BP-1885270558-172.28.0.12-1789898005200:blk_1073741845_1021 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-41090938-09f6-46ff-be46-644b2c83f621,DISK]]
1. BP-1885270558-172.28.0.12-1789898005200:blk_1073741846_1022 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-41090938-09f6-46ff-be46-644b2c83f621,DISK]]
2. BP-1885270558-172.28.0.12-1789898005200:blk_1073741847_1023 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-41090938-09f6-46ff-be46-644b2c83f621,DISK]]
3. BP-1885270558-172.28.0.12-1789898005200:blk_1073741848_1024 len=134217728 Live

In [29]:
# Step 19: Check the default HDFS block size.

!hdfs getconf -confKey dfs.blocksize

134217728


In [30]:
# Step 20: Check the configured replication factor.

!hdfs getconf -confKey dfs.replication

1


In [31]:
#!hdfs dfs -cat /credit/credit_card_transactions-ibm_v2.csv

In [32]:
file_path = "/content/extracted_folder/credit_card_transactions-ibm_v2.csv"

df = pd.read_csv(file_path)

print(df.shape)
display(df.head())
df.info()

(24386900, 15)


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   User            int64  
 1   Card            int64  
 2   Year            int64  
 3   Month           int64  
 4   Day             int64  
 5   Time            object 
 6   Amount          object 
 7   Use Chip        object 
 8   Merchant Name   int64  
 9   Merchant City   object 
 10  Merchant State  object 
 11  Zip             float64
 12  MCC             int64  
 13  Errors?         object 
 14  Is Fraud?       object 
dtypes: float64(1), int64(7), object(7)
memory usage: 2.7+ GB


Columns Name: User, Card, Year, Month, Day, Time, Amount, Use Chip,
Merchant Name, Merchant City, Merchant State, Zip, MCC,
Errors?, Is Fraud?

Cleaning Process:
- conveting time into the date and time format
- In column amount, using regex remove $ sign, and convert into numeric
- Converting Is Fraud? into binary (1 and 0)

## Analysis Part

In [33]:
!pip install -q pyspark==3.5.6

We are going to ignore this error because there is need to download version == 3.5.6 to solve the bugs.

In [34]:
import os
import shutil
import subprocess

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("Java executable:", shutil.which("java"))

print(subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
).stderr)

JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
Java executable: /usr/lib/jvm/java-11-openjdk-amd64/bin/java
openjdk version "11.0.32" 2026-07-21
OpenJDK Runtime Environment (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)



In [35]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CreditCardFraudDetection")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.6


In [36]:
!hdfs dfs -ls /credit

Found 2 items
-rw-r--r--   1 root supergroup 2350744057 2026-09-20 10:23 /credit/credit_card_transactions-ibm_v2.csv
-rw-r--r--   1 root supergroup  208414209 2026-09-20 10:10 /credit/credit_transactions.parquet


In [37]:
!hdfs dfs -get -f /credit/credit_card_transactions-ibm_v2.csv /content/

In [38]:
hdfs_path = "hdfs://localhost:9000/credit/credit_card_transactions-ibm_v2.csv"

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(hdfs_path)
)

print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))

raw_df.printSchema()
raw_df.show(5, truncate=False)

Rows: 24386900
Columns: 15
root
 |-- User: integer (nullable = true)
 |-- Card: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Amount: string (nullable = true)
 |-- Use Chip: string (nullable = true)
 |-- Merchant Name: long (nullable = true)
 |-- Merchant City: string (nullable = true)
 |-- Merchant State: string (nullable = true)
 |-- Zip: double (nullable = true)
 |-- MCC: integer (nullable = true)
 |-- Errors?: string (nullable = true)
 |-- Is Fraud?: string (nullable = true)

+----+----+----+-----+---+-------------------+-------+-----------------+-------------------+-------------+--------------+-------+----+-------+---------+
|User|Card|Year|Month|Day|Time               |Amount |Use Chip         |Merchant Name      |Merchant City|Merchant State|Zip    |MCC |Errors?|Is Fraud?|
+----+----+----+-----+---+-------------------+-------+-----------------+--

In case if schema is not able to detect the correct datatype, read everything as string and cast columns explicitly later.


In [39]:
# showing the tail values
tail_data = raw_df.tail(5)

# Convert to a DataFrame and show it in a table
spark.createDataFrame(tail_data, raw_df.schema).show()


+----+----+----+-----+---+-------------------+-------+----------------+--------------------+-------------+--------------+------+----+-------+---------+
|User|Card|Year|Month|Day|               Time| Amount|        Use Chip|       Merchant Name|Merchant City|Merchant State|   Zip| MCC|Errors?|Is Fraud?|
+----+----+----+-----+---+-------------------+-------+----------------+--------------------+-------------+--------------+------+----+-------+---------+
|1999|   1|2020|    2| 27|2026-09-20 22:23:00|$-54.00|Chip Transaction|-5162038175624867091|    Merrimack|            NH|3054.0|5541|   NULL|       No|
|1999|   1|2020|    2| 27|2026-09-20 22:24:00| $54.00|Chip Transaction|-5162038175624867091|    Merrimack|            NH|3054.0|5541|   NULL|       No|
|1999|   1|2020|    2| 28|2026-09-20 07:43:00| $59.15|Chip Transaction| 2500998799892805156|    Merrimack|            NH|3054.0|4121|   NULL|       No|
|1999|   1|2020|    2| 28|2026-09-20 20:10:00| $43.12|Chip Transaction| 2500998799892805

In [40]:
raw_df.describe().show()

+-------+------------------+------------------+------------------+------------------+------------------+--------+-----------------+--------------------+-------------+--------------+------------------+-----------------+----------------+---------+
|summary|              User|              Card|              Year|             Month|               Day|  Amount|         Use Chip|       Merchant Name|Merchant City|Merchant State|               Zip|              MCC|         Errors?|Is Fraud?|
+-------+------------------+------------------+------------------+------------------+------------------+--------+-----------------+--------------------+-------------+--------------+------------------+-----------------+----------------+---------+
|  count|          24386900|          24386900|          24386900|          24386900|          24386900|24386900|         24386900|            24386900|     24386900|      21666079|          21508765|         24386900|          388431| 24386900|
|   mean|1001.01

In [41]:
from pyspark.sql.functions import col, sum as spark_sum

missing_df = raw_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in raw_df.columns
])

missing_df.show()

+----+----+----+-----+---+----+------+--------+-------------+-------------+--------------+-------+---+--------+---------+
|User|Card|Year|Month|Day|Time|Amount|Use Chip|Merchant Name|Merchant City|Merchant State|    Zip|MCC| Errors?|Is Fraud?|
+----+----+----+-----+---+----+------+--------+-------------+-------------+--------------+-------+---+--------+---------+
|   0|   0|   0|    0|  0|   0|     0|       0|            0|            0|       2720821|2878135|  0|23998469|        0|
+----+----+----+-----+---+----+------+--------+-------------+-------------+--------------+-------+---+--------+---------+



### Data Description

|||
|----|----|
|Number of Rows (/Users)|24386900|
|Number of Columns|15|
|Year Range | 1991 - 2020|

Necessary Changes:
- Convert the data type of the `Amount` column.
- There is null values into the


In [42]:
from pyspark.sql.functions import regexp_replace, col

# Replace 'Amount' with your actual column name
raw_df = raw_df.withColumn("Amount", regexp_replace(col("Amount"), r"\$", "").cast("float"))

raw_df.show(5)
# data type of column amount
raw_df.printSchema()


+----+----+----+-----+---+-------------------+------+-----------------+-------------------+-------------+--------------+-------+----+-------+---------+
|User|Card|Year|Month|Day|               Time|Amount|         Use Chip|      Merchant Name|Merchant City|Merchant State|    Zip| MCC|Errors?|Is Fraud?|
+----+----+----+-----+---+-------------------+------+-----------------+-------------------+-------------+--------------+-------+----+-------+---------+
|   0|   0|2002|    9|  1|2026-09-20 06:21:00|134.09|Swipe Transaction|3527213246127876953|     La Verne|            CA|91750.0|5300|   NULL|       No|
|   0|   0|2002|    9|  1|2026-09-20 06:42:00| 38.48|Swipe Transaction|-727612092139916043|Monterey Park|            CA|91754.0|5411|   NULL|       No|
|   0|   0|2002|    9|  2|2026-09-20 06:22:00|120.34|Swipe Transaction|-727612092139916043|Monterey Park|            CA|91754.0|5411|   NULL|       No|
|   0|   0|2002|    9|  2|2026-09-20 17:45:00|128.95|Swipe Transaction|34145274595791067

In [43]:
# check the fraud target

raw_df.groupBy("Is Fraud?").count().orderBy("count", ascending=False).show()

+---------+--------+
|Is Fraud?|   count|
+---------+--------+
|       No|24357143|
|      Yes|   29757|
+---------+--------+



In [44]:
# check fraud percentage

from pyspark.sql.functions import round

fraud_summary = (
    raw_df.groupBy("Is Fraud?")
    .count()
    .withColumn(
        "percentage",
        round(col("count") / raw_df.count() * 100, 4)
    )
)

fraud_summary.show()

+---------+--------+----------+
|Is Fraud?|   count|percentage|
+---------+--------+----------+
|       No|24357143|    99.878|
|      Yes|   29757|     0.122|
+---------+--------+----------+



In [ ]:
raw_df.select("Amount").show(10)

raw_df.groupBy("Is Fraud?").agg(
    spark_sum("Amount").alias("total_amount")
).show()

+------+
|Amount|
+------+
|134.09|
| 38.48|
|120.34|
|128.95|
|104.71|
| 86.19|
| 93.84|
| 123.5|
| 61.72|
|  57.1|
+------+
only showing top 10 rows



In [ ]:
# showing 10 values only if 'Is Fraud?' is NO.
raw_df.filter(col("Is Fraud?") == "No").select("Amount").show(10)

# showing 10 values only if 'Is Fraud?' is YES.
raw_df.filter(col("Is Fraud?") == "Yes").select("Amount").show(10)

In [ ]:
from pyspark.sql.functions import (
    trim, lower, when, regexp_replace, to_timestamp,
    hour, coalesce, lit
)

df = raw_df

# Again Clean Amount in case it contains currency symbols or commas
df = df.withColumn(
    "Amount",
    regexp_replace(
        regexp_replace(trim(col("Amount").cast("string")), "[$,]", ""),
        " ",
        ""
    ).cast("double")
)

# Cast numeric fields
numeric_columns = ["Year", "Month", "Day", "Zip", "MCC"]

for column_name in numeric_columns:
    df = df.withColumn(
        column_name,
        col(column_name).cast("double")
    )

# Normalize target values
df = df.withColumn(
    "fraud_label",
    when(
        lower(trim(col("Is Fraud?").cast("string"))).isin(
            "yes", "true", "1", "fraud"
        ),
        1.0
    ).otherwise(0.0)
)

In [ ]:
df = df.withColumn(
    "transaction_hour",
    hour(
        to_timestamp(trim(col("Time")), "HH:mm:ss")
        )
)

In [ ]:
df.show(5)

In [ ]:
# checkinng the null values after parsing
missing_df1 = raw_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in raw_df.columns
])

missing_df1.show()

In [ ]:
# clean categorical variables
categorical_columns = [
    "Use Chip",
    "Merchant City",
    "Merchant State",
    "Errors?"
]

for column_name in categorical_columns:
    df = df.withColumn(
        column_name,
        when(
            col(column_name).isNull() |
            (trim(col(column_name).cast("string")) == ""),
            "Unknown"
        ).otherwise(trim(col(column_name).cast("string")))
    )

In [ ]:
df = df.dropna(
    subset=["Amount", "Year", "Month", "Day", "MCC", "fraud_label"]
)

print("Clean rows:", df.count())
df.select(
    "Amount",
    "Year",
    "Month",
    "Day",
    "transaction_hour",
    "MCC",
    "fraud_label"
).show(5)

All the rows are cleaned.

In [ ]:
# Fraud rate by Month
from pyspark.sql.functions import avg, count

df.groupBy("Month").agg(
    count("*").alias("transactions"),
    avg("fraud_label").alias("fraud_rate")
).orderBy("Month").show()

In [ ]:
# Fraud rate by transaction Hours
df.groupBy("transaction_hour").agg(
    count("*").alias("transactions"),
    avg("fraud_label").alias("fraud_rate")
).orderBy("transaction_hour").show()

In [ ]:
# Fraud rate by Payment method
df.groupBy("Use Chip").agg(
    count("*").alias("transactions"),
    avg("fraud_label").alias("fraud_rate")
).orderBy(col("fraud_rate").desc()).show()

In [ ]:
mondthly_df = df.groupby("Month").agg(avg("fraud_label").alias("fraud_rate")).orderBy("Month")
mondthly_df.show()

In [ ]:
# Visulising the aggrigate results

monthly_pd = (
    df.groupBy("Month")
    .agg(avg("fraud_label").alias("fraud_rate"))
    .orderBy("Month")
    .toPandas()
)

monthly_pd.plot(
    x="Month",
    y="fraud_rate",
    kind="bar",
    legend=False,
    figsize=(8, 4)
)

plt.title("Fraud Rate by Month")
plt.ylabel("Fraud Rate")
plt.show()